In [1]:
import time
import boto3
import sagemaker
import json
import numpy as np
from sagemaker import get_execution_role
from sagemaker.amazon.amazon_estimator import get_image_uri
from sagemaker.tuner import IntegerParameter, CategoricalParameter, ContinuousParameter, HyperparameterTuner
from sagemaker.model import Model
from sagemaker.predictor import RealTimePredictor

# Configuring the built-in Image Classification algorithm

## Create a unique job name 

In [2]:
job_name_prefix = 'breast-cancer-detection'
timestamp = time.strftime('-%Y-%m-%d-%H-%M-%S', time.gmtime())
job_name = job_name_prefix + timestamp

## Specify the input and output paths for the job

In [3]:
bucket = 'alexbc.datalake.us-east-2'
input_prefix = 'breast-cancer-histopathology/curated-data/recordio'
input_train = 's3://{}/{}/train/'.format(bucket, input_prefix)
input_test = 's3://{}/{}/test/'.format(bucket, input_prefix)

output_prefix = 'breast-cancer-histopathology/model-output/sagemaker-api'
output_path = 's3://{}/{}/'.format(bucket, output_prefix)

## Creating an Estimator to build image classification

In [4]:
role = get_execution_role()
training_image = get_image_uri(boto3.Session().region_name, 'image-classification')

sagemaker_session = sagemaker.Session()
estimator = sagemaker.estimator.Estimator(training_image, 
                                          role, 
                                          train_instance_count=1,
                                          train_instance_type='ml.p2.xlarge',
                                          train_volume_size=50,
                                          train_max_run=360000,
                                          output_path=output_path, 
                                          sagemaker_session=sagemaker_session,
                                          input_mode='Pipe')

In [5]:
estimator.set_hyperparameters(num_classes=2,
                              num_training_samples=194266,
                              num_layers=18,
                              mini_batch_size=128,
                              image_shape='3,50,50',
                              augmentation_type='crop_color_transform',
                              epochs=5,
                              learning_rate=0.01,
                              use_pretrained_model=1)

# Create a training job

In [6]:
s3_input_train = sagemaker.s3_input(s3_data=input_train, content_type='application/x-recordio')
s3_input_validation = sagemaker.s3_input(s3_data=input_test, content_type='application/x-recordio')

In [7]:
estimator.fit({
    'train': s3_input_train,
    'validation': s3_input_validation
}, job_name=job_name)

2020-02-01 15:59:41 Starting - Starting the training job...
2020-02-01 15:59:42 Starting - Launching requested ML instances...
2020-02-01 16:00:37 Starting - Preparing the instances for training.........
2020-02-01 16:01:58 Downloading - Downloading input data...
2020-02-01 16:02:35 Training - Downloading the training image......
2020-02-01 16:03:35 Training - Training image download completed. Training in progress.Docker entrypoint called with argument(s): train
[02/01/2020 16:03:38 INFO 140600552224576] Reading default configuration from /opt/amazon/lib/python2.7/site-packages/image_classification/default-input.json: {u'beta_1': 0.9, u'gamma': 0.9, u'beta_2': 0.999, u'optimizer': u'sgd', u'use_pretrained_model': 0, u'eps': 1e-08, u'epochs': 30, u'lr_scheduler_factor': 0.1, u'num_layers': 152, u'image_shape': u'3,224,224', u'precision_dtype': u'float32', u'mini_batch_size': 32, u'weight_decay': 0.0001, u'learning_rate': 0.1, u'momentum': 0}
[02/01/2020 16:03:38 INFO 140600552224576] M

# Deploying the model generated by the training job

## Create a unique model name

In [8]:
model_name_prefix = 'bcd-image-classification-high-level'
model_artifacts_s3_path = 's3://alexbc.datalake.us-east-2/breast-cancer-histopathology/model-output/sagemaker-api/breast-cancer-detection-2020-02-01-15-55-04/output/model.tar.gz'
hosting_image = get_image_uri(boto3.Session().region_name, 'image-classification')

timestamp = time.strftime('-%Y-%m-%d-%H-%M-%S', time.gmtime())
model_name = model_name_prefix + timestamp

## Create a Model object

In [9]:
model = Model(
    name=model_name,
    model_data=model_artifacts_s3_path,
    image=hosting_image,
    role=role,
    predictor_cls=lambda endpoint_name, sagemaker_session: RealTimePredictor(endpoint_name, sagemaker_session)
)

## Create a unique endpoint name

In [10]:
endpoint_name_prefix = 'breast-cancer-detection-ep'
timestamp = time.strftime('-%Y-%m-%d-%H-%M-%S', time.gmtime())
endpoint_name = endpoint_name_prefix + timestamp

## Create a model, an endpoint configuration and an endpoint

In [12]:
predictor = model.deploy(
    endpoint_name=endpoint_name,
    initial_instance_count=1,
    instance_type='ml.m4.xlarge'
)

---------------!

# Testing the deployed model

In [18]:
def predict_breast_cancer(image_path):
    with open(image_path, 'rb') as f:
        payload = f.read()
        payload = bytearray(payload)
    response = predictor.predict(payload)
    result = json.loads(response)
    print('Probabilities for all classes: ', result)
    predicted_class = np.argmax(result)
    if predicted_class == 0:
        print('Breast cancer not detected')
    else:
        print('Breast cancer detected')

In [19]:
predict_breast_cancer('8975_idx5_x2851_y1201_class0.png')

Probabilities for all classes:  [0.8581951856613159, 0.14180487394332886]
Breast cancer not detected


In [20]:
predict_breast_cancer('10253_idx5_x501_y401_class1.png')

Probabilities for all classes:  [0.390950471162796, 0.6090494990348816]
Breast cancer detected


# Deleting endpoint

In [ ]:
sagemaker.Session().delete_endpoint(predictor.endpoint)